# 03D — Classe CATMAT: BPS × catálogo oficial (SOMENTE MEDE)
**TCC — Preços e concentração de fornecedores em compras públicas de insumos hospitalares, 2009–2023**

**Decisão D40 (14/10/2026).** O tipo de produto é a **classe CATMAT do governo**, com o nome oficial, sem agrupamento do trabalho.
A classe de cada código vem em duas etapas: (1) a do próprio BPS, que só traz a classe nos arquivos de 2018–2020;
(2) para os códigos que ficarem sem classe, o **catálogo oficial de materiais** (Compras.gov.br, dados abertos, módulo Material).
A etapa 2 só será aplicada depois desta medição.

**Esta rotina não altera nenhuma base.** Ela:
1. monta a classe que o BPS informa (2018–2020) e confere se algum código aparece em mais de uma classe;
2. baixa do catálogo oficial a classe de todos os códigos da base pública;
3. **conferência:** nos códigos que têm classe no BPS, compara a classe do BPS com a do catálogo;
4. mede quanto a etapa 2 cobriria: códigos e compras sem classe, por ano, antes e depois.

**Lastro.** O BPS adota a codificação do CATMAT (Ministério da Saúde, página "CATMAT"). Classes e nomes oficiais:
Ministério da Saúde, "Passo a passo do BPS: consulta de itens". Consulta: Manual da API de Dados Abertos do Compras.gov.br,
endpoint `modulo-material/4_consultarItemMaterial`.

**Limite.** O catálogo mostra a classe de hoje, não a da época da compra; a conferência (passo 3) mede se isso faz diferença.

**Entradas:** `Base Harmonizacao Publica`, `Base Harmonizacao Campos` (2018–2020).
**Saídas:** `Base Classificacao CATMAT/` (o que o catálogo devolveu) e `Auditoria/03d`, `03e`, `03f`, `03g`.
**Precisa de internet** (rodar no Colab).

In [1]:
# ── CAMINHO DOS DADOS ────────────────────────────────────────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass   # fora do Colab: use a variável de ambiente BPS_DADOS
PASTA_DADOS = os.environ.get('BPS_DADOS', '/content/drive/MyDrive/TCC/Registro de Compras v2/Base v2')


Mounted at /content/drive


In [2]:
# ─────────────────────────────────────────────────────────────────────────
!pip install polars==1.35.2 pyarrow requests --quiet
import time, json
import polars as pl
import requests
from pathlib import Path

BASE            = Path(PASTA_DADOS)
P_PUB           = BASE / 'Base Harmonizacao Publica'
P_HARM          = BASE / 'Base Harmonizacao Campos'
PASTA_CATALOGO  = BASE / 'Base Classificacao CATMAT'
PASTA_AUDITORIA = BASE / 'Auditoria'
for p in (PASTA_CATALOGO, PASTA_AUDITORIA):
    p.mkdir(parents=True, exist_ok=True)
ANO_INICIO, ANO_FIM = 2009, 2023
ANOS_COM_CLASSE_NO_BPS = (2018, 2019, 2020)   # únicos arquivos do BPS com a coluna de classe (rotina 11, seção 11)
print('Pronto ✓')

Pronto ✓


## 1. Códigos da base pública e classe informada pelo BPS (etapa 1)

In [3]:
pub = pl.concat([pl.read_parquet(a, columns=['cod_catmat', 'ano', 'desc_item'])
                 for a in sorted(P_PUB.glob('BPS_*.parquet'))], how='diagonal')
pub = pub.filter(pl.col('ano').is_between(ANO_INICIO, ANO_FIM) & pl.col('cod_catmat').is_not_null())
print(f'Compras na base pública {ANO_INICIO}-{ANO_FIM} com código: {pub.height:,} | códigos distintos: {pub["cod_catmat"].n_unique():,}')

classe_bps = (pl.concat([pl.read_parquet(P_HARM / f'BPS_{a}.parquet', columns=['cod_catmat', 'classe_catmat'])
                         for a in ANOS_COM_CLASSE_NO_BPS])
                .with_columns(pl.col('classe_catmat').cast(pl.Utf8).str.strip_chars())
                .filter(pl.col('cod_catmat').is_not_null() & pl.col('classe_catmat').is_not_null() & (pl.col('classe_catmat') != ''))
                .select('cod_catmat', pl.col('classe_catmat').alias('classe_bps')).unique())
amb = classe_bps.group_by('cod_catmat').len().filter(pl.col('len') > 1)
print(f'Códigos com classe no BPS ({ANOS_COM_CLASSE_NO_BPS[0]}-{ANOS_COM_CLASSE_NO_BPS[-1]}): {classe_bps["cod_catmat"].n_unique():,} '
      f'| em mais de uma classe: {amb.height}')
assert amb.height == 0, 'Há códigos com mais de uma classe no BPS: decidir antes de seguir'

codigos = pub.select('cod_catmat').unique().join(classe_bps, on='cod_catmat', how='left')
print(f'Códigos da base pública sem classe no BPS: {codigos.filter(pl.col("classe_bps").is_null()).height:,}')

Compras na base pública 2009-2023 com código: 836,096 | códigos distintos: 15,955
Códigos com classe no BPS (2018-2020): 10,887 | em mais de uma classe: 0
Códigos da base pública sem classe no BPS: 5,118


## 2. Classe no catálogo oficial (Compras.gov.br)
> Consulta código a código no endpoint `4_consultarItemMaterial` (parâmetro `codigoItem`). O código do BPS é "BR" + 7 dígitos;
> no catálogo o mesmo código vai sem o "BR" e sem os zeros à esquerda (ex.: BR0272335 → 272335).
>
> **Retomada:** cada resposta é guardada em `Base Classificacao CATMAT/consultas_catalogo.jsonl`. Se a execução cair
> (internet, Colab desconectado), basta rodar a célula de novo: ela pula os códigos já consultados.

In [4]:
URL_ITEM = 'https://dadosabertos.compras.gov.br/modulo-material/4_consultarItemMaterial'
ARQ_CACHE = PASTA_CATALOGO / 'consultas_catalogo.jsonl'
# Parâmetros técnicos da consulta (D40). Não afetam nenhum resultado, só a forma de baixar:
TEMPO_LIMITE_S   = 30    # espera máxima por resposta antes de tentar de novo
TENTATIVAS       = 5     # tentativas por código antes de registrar falha
CONSULTAS_JUNTAS = 1     # consultas simultâneas. Na 1ª execução (14/10/2026) foram 4 e o serviço recusou 607 códigos
                         # com "429 - muitas consultas"; com 1 por vez a repetição respeita o limite do serviço
ESPERA_429_S     = 30    # espera após um "429 - muitas consultas" (multiplicada pela tentativa), se o serviço não disser quanto esperar
GRAVAR_A_CADA    = 200   # grava o progresso no arquivo a cada N consultas (para poder retomar)

def para_codigo_catalogo(cod):
    return int(cod.replace('BR', ''))

def consultar(codigo_item):
    for t in range(TENTATIVAS):
        try:
            r = requests.get(URL_ITEM, params={'pagina': 1, 'codigoItem': codigo_item}, timeout=TEMPO_LIMITE_S)
            if r.status_code == 200:
                return {'codigoItem': codigo_item, 'http': 200, 'resposta': r.json()}
            ultimo = r.status_code
            if r.status_code == 429:   # o serviço pediu para diminuir o ritmo
                pedido = r.headers.get('Retry-After', '')
                time.sleep(int(pedido) if pedido.isdigit() else ESPERA_429_S * (t + 1))
                continue
        except Exception as e:
            ultimo = f'erro: {type(e).__name__}'
        time.sleep(2 ** t)   # espera crescente entre tentativas (1, 2, 4, 8, 16 s)
    return {'codigoItem': codigo_item, 'http': ultimo, 'resposta': None}

ja = set()
if ARQ_CACHE.exists():
    with open(ARQ_CACHE) as f:
        for linha in f:
            reg = json.loads(linha)
            if reg['http'] == 200:
                ja.add(reg['codigoItem'])
faltam = [c for c in (para_codigo_catalogo(x) for x in codigos['cod_catmat'].to_list()) if c not in ja]
print(f'Códigos a consultar: {len(faltam):,} (já consultados antes: {len(ja):,})')
# a internet só é usada se faltar algum código; com todas as respostas guardadas
# (consultas_catalogo.jsonl, out/2026, D41), a rotina roda sem acesso ao serviço
if faltam:
    # teste do serviço antes de começar: BR0272335 tem classe 6505 no BPS de 2018
    teste = consultar(272335)
    assert teste['http'] == 200, f'Catálogo não respondeu ({teste["http"]}); tente mais tarde'
    print('Teste do catálogo: BR0272335 ->', [(x['codigoItem'], x['codigoClasse'], x['nomeClasse']) for x in teste['resposta']['resultado']])
else:
    print('Todas as respostas já estão guardadas: nenhuma consulta ao serviço.')

from concurrent.futures import ThreadPoolExecutor
inicio = time.time()
for ini in range(0, len(faltam), GRAVAR_A_CADA):
    lote = faltam[ini:ini + GRAVAR_A_CADA]
    with ThreadPoolExecutor(max_workers=CONSULTAS_JUNTAS) as ex:
        resultados = list(ex.map(consultar, lote))
    with open(ARQ_CACHE, 'a') as f:
        for reg in resultados:
            f.write(json.dumps(reg, ensure_ascii=False) + '\n')
    feitos = ini + len(lote)
    print(f'  {feitos:,}/{len(faltam):,} consultados | {(time.time() - inicio) / 60:.1f} min')

Códigos a consultar: 0 (já consultados antes: 15,955)
Todas as respostas já estão guardadas: nenhuma consulta ao serviço.


In [5]:
# Lê o que o catálogo devolveu (fica a última resposta de cada código)
regs = {}
with open(ARQ_CACHE) as f:
    for linha in f:
        reg = json.loads(linha)
        regs[reg['codigoItem']] = reg
linhas, falhas = [], []
for c, reg in regs.items():
    if reg['http'] != 200:
        falhas.append({'codigoItem': c, 'http': str(reg['http'])})
        linhas.append({'codigoItem': c, 'consulta': f'falhou ({reg["http"]})'}); continue
    res = [x for x in (reg['resposta'] or {}).get('resultado', []) if x.get('codigoItem') == c]
    if not res:
        linhas.append({'codigoItem': c, 'consulta': 'ok', 'encontrado': False}); continue
    x = res[0]
    linhas.append({'codigoItem': c, 'consulta': 'ok', 'encontrado': True, 'classe_catalogo': str(x.get('codigoClasse')),
                   'nome_classe': x.get('nomeClasse'), 'grupo': str(x.get('codigoGrupo')), 'nome_grupo': x.get('nomeGrupo'),
                   'item_ativo': x.get('statusItem'), 'descricao_catalogo': x.get('descricaoItem'),
                   'atualizado_em': x.get('dataHoraAtualizacao'), 'respostas_do_codigo': len(res)})
catalogo = pl.DataFrame(linhas, infer_schema_length=None)
ok = catalogo.filter(pl.col('consulta') == 'ok')
print(f'Consultados com resposta: {ok.height:,} | encontrados no catálogo: {ok.filter(pl.col("encontrado")).height:,} '
      f'| não encontrados: {ok.filter(~pl.col("encontrado")).height:,} '
      f'| falhas de consulta: {len(falhas)}')
if falhas:
    print('ATENÇÃO: há códigos sem resposta do serviço; rode a célula anterior de novo para tentar só esses.')
if 'respostas_do_codigo' in catalogo.columns:
    assert catalogo.filter(pl.col('respostas_do_codigo') > 1).height == 0, 'Catálogo devolveu mais de um registro para o mesmo código'
catalogo = catalogo.with_columns(pl.format('BR{}', pl.col('codigoItem').cast(pl.Utf8).str.pad_start(7, '0')).alias('cod_catmat'))
catalogo.write_parquet(PASTA_CATALOGO / 'classe_catalogo.parquet')
catalogo.write_csv(PASTA_CATALOGO / 'classe_catalogo.csv')
print('Gravado: Base Classificacao CATMAT/classe_catalogo.parquet / .csv')

Consultados com resposta: 15,955 | encontrados no catálogo: 15,955 | não encontrados: 0 | falhas de consulta: 0
Gravado: Base Classificacao CATMAT/classe_catalogo.parquet / .csv


## 3. Conferência: classe do BPS × classe do catálogo
> Nos códigos que têm classe no BPS (2018–2020), a classe do catálogo de hoje é a mesma?
> É o que diz se o catálogo pode preencher os códigos que o BPS deixou sem classe (D40).

In [6]:
conf = codigos.join(catalogo.select('cod_catmat', 'encontrado', 'classe_catalogo', 'nome_classe', 'item_ativo'),
                    on='cod_catmat', how='left')
comparaveis = conf.filter(pl.col('classe_bps').is_not_null() & pl.col('encontrado').fill_null(False))
iguais = comparaveis.filter(pl.col('classe_bps') == pl.col('classe_catalogo'))
compras_cod = pub.group_by('cod_catmat').len().rename({'len': 'compras'})
comp_c = comparaveis.join(compras_cod, on='cod_catmat')
ig_c = comp_c.filter(pl.col('classe_bps') == pl.col('classe_catalogo'))

t03d = pl.DataFrame({
    'medida': ['codigos com classe no BPS', 'encontrados no catalogo', 'mesma classe', 'classe diferente'],
    'codigos': [conf.filter(pl.col('classe_bps').is_not_null()).height, comparaveis.height, iguais.height,
                comparaveis.height - iguais.height],
    'compras': [conf.filter(pl.col('classe_bps').is_not_null()).join(compras_cod, on='cod_catmat')['compras'].sum(),
                comp_c['compras'].sum(), ig_c['compras'].sum(), comp_c['compras'].sum() - ig_c['compras'].sum()]})
print(t03d)
print(f'\nCONCORDÂNCIA: mesma classe em {iguais.height:,} de {comparaveis.height:,} códigos = {iguais.height / comparaveis.height:.2%} '
      f'| em compras: {ig_c["compras"].sum() / comp_c["compras"].sum():.2%}')
t03d.write_csv(PASTA_AUDITORIA / '03d_classe_bps_x_catalogo.csv')

diverg = (comp_c.filter(pl.col('classe_bps') != pl.col('classe_catalogo'))
            .join(pub.group_by('cod_catmat').agg(pl.col('desc_item').drop_nulls().last().alias('descricao_bps')), on='cod_catmat')
            .select('cod_catmat', 'classe_bps', 'classe_catalogo', 'nome_classe', 'item_ativo', 'compras', 'descricao_bps')
            .sort('compras', descending=True))
print('\nCódigos com classe diferente (maiores em compras):'); print(diverg.head(20))
print('\nDe que classe para que classe:')
print(diverg.group_by('classe_bps', 'classe_catalogo').agg(pl.len().alias('codigos'), pl.col('compras').sum()).sort('compras', descending=True))
diverg.write_csv(PASTA_AUDITORIA / '03e_divergencias_classe.csv')
print('\nRegistros do teste: Auditoria/03d_classe_bps_x_catalogo.csv e 03e_divergencias_classe.csv')

shape: (4, 3)
┌───────────────────────────┬─────────┬─────────┐
│ medida                    ┆ codigos ┆ compras │
│ ---                       ┆ ---     ┆ ---     │
│ str                       ┆ i64     ┆ i64     │
╞═══════════════════════════╪═════════╪═════════╡
│ codigos com classe no BPS ┆ 10837   ┆ 808021  │
│ encontrados no catalogo   ┆ 10837   ┆ 808021  │
│ mesma classe              ┆ 10821   ┆ 807587  │
│ classe diferente          ┆ 16      ┆ 434     │
└───────────────────────────┴─────────┴─────────┘

CONCORDÂNCIA: mesma classe em 10,821 de 10,837 códigos = 99.85% | em compras: 99.95%

Códigos com classe diferente (maiores em compras):
shape: (16, 7)
┌────────────┬────────────┬────────────────┬────────────────┬────────────┬─────────┬───────────────┐
│ cod_catmat ┆ classe_bps ┆ classe_catalog ┆ nome_classe    ┆ item_ativo ┆ compras ┆ descricao_bps │
│ ---        ┆ ---        ┆ o              ┆ ---            ┆ ---        ┆ ---     ┆ ---           │
│ str        ┆ str        ┆ --

## 4. Quanto a etapa 2 cobriria (por ano)
> Para cada ano: compras com classe só pelo BPS, e com BPS + catálogo nos códigos que o BPS deixou sem classe.
> Nada é aplicado aqui.

In [7]:
cls = conf.select('cod_catmat', 'classe_bps',
                  pl.when(pl.col('classe_bps').is_null() & pl.col('encontrado').fill_null(False))
                    .then(pl.col('classe_catalogo')).alias('classe_so_catalogo'))
t03f = (pub.join(cls, on='cod_catmat', how='left').group_by('ano')
           .agg(pl.len().alias('compras'),
                pl.col('classe_bps').is_not_null().sum().alias('com_classe_bps'),
                (pl.col('classe_bps').is_not_null() | pl.col('classe_so_catalogo').is_not_null()).sum().alias('com_classe_bps_ou_catalogo'))
           .with_columns((pl.col('com_classe_bps') / pl.col('compras') * 100).round(1).alias('pct_bps'),
                         (pl.col('com_classe_bps_ou_catalogo') / pl.col('compras') * 100).round(1).alias('pct_bps_ou_catalogo'),
                         (pl.col('compras') - pl.col('com_classe_bps_ou_catalogo')).alias('continuariam_sem_classe'))
           .sort('ano'))
print(t03f)
t03f.write_csv(PASTA_AUDITORIA / '03f_cobertura_classe_por_ano.csv')

sem = conf.filter(pl.col('classe_bps').is_null())
print(f'\nCódigos sem classe no BPS: {sem.height:,} | o catálogo encontra: {sem.filter(pl.col("encontrado").fill_null(False)).height:,} '
      f'| o catálogo não encontra: {sem.filter(pl.col("encontrado") == False).height:,} '
      f'| consulta falhou (rodar a seção 2 de novo): {sem.filter(pl.col("encontrado").is_null()).height:,}')
print('\nClasse que o catálogo daria aos códigos sem classe no BPS:')
print(sem.filter(pl.col('encontrado').fill_null(False)).join(compras_cod, on='cod_catmat')
         .group_by('classe_catalogo', 'nome_classe').agg(pl.len().alias('codigos'), pl.col('compras').sum())
         .sort('compras', descending=True))
print('\nRegistro: Auditoria/03f_cobertura_classe_por_ano.csv (nenhuma base de análise foi alterada)')

shape: (15, 7)
┌──────┬─────────┬────────────────┬──────────────────┬─────────┬─────────────────┬─────────────────┐
│ ano  ┆ compras ┆ com_classe_bps ┆ com_classe_bps_o ┆ pct_bps ┆ pct_bps_ou_cata ┆ continuariam_se │
│ ---  ┆ ---     ┆ ---            ┆ u_catalogo       ┆ ---     ┆ logo            ┆ m_classe        │
│ i32  ┆ u32     ┆ u32            ┆ ---              ┆ f64     ┆ ---             ┆ ---             │
│      ┆         ┆                ┆ u32              ┆         ┆ f64             ┆ u32             │
╞══════╪═════════╪════════════════╪══════════════════╪═════════╪═════════════════╪═════════════════╡
│ 2009 ┆ 5111    ┆ 4453           ┆ 5111             ┆ 87.1    ┆ 100.0           ┆ 0               │
│ 2010 ┆ 10072   ┆ 9118           ┆ 10072            ┆ 90.5    ┆ 100.0           ┆ 0               │
│ 2011 ┆ 8306    ┆ 7489           ┆ 8306             ┆ 90.2    ┆ 100.0           ┆ 0               │
│ 2012 ┆ 3217    ┆ 2794           ┆ 3217             ┆ 86.9    ┆ 100.0      

## 5. Códigos fora das classes de saúde (D41)
> A D41 tira da base os códigos cuja classe está fora das 16 classes de saúde publicadas pelo Ministério da Saúde
> ("Passo a passo do BPS: consulta de itens"). A regra é pela classe, nunca item a item. Esta seção só lista
> os códigos que sairão, com a descrição, para conferência; a retirada é feita na base analítica.

In [8]:
# Classes de saúde do Ministério da Saúde (D41): lista publicada em "Passo a passo do BPS: consulta de itens"
CLASSES_SAUDE_MS = ['6505', '6508', '6509', '6510', '6515', '6520', '6525', '6530', '6532',
                    '6540', '6545', '6550', '6640', '6810', '6840', '8940']
classe_final = conf.select('cod_catmat',
                           pl.coalesce('classe_bps', 'classe_catalogo').alias('classe'),
                           pl.when(pl.col('classe_bps').is_not_null()).then(pl.lit('BPS 2018-2020'))
                             .otherwise(pl.lit('catalogo, consulta out/2026')).alias('origem_classe'),
                           'item_ativo')
fora = (classe_final.filter(~pl.col('classe').is_in(CLASSES_SAUDE_MS))
          .join(catalogo.select('cod_catmat', 'nome_classe'), on='cod_catmat', how='left')
          .join(compras_cod, on='cod_catmat')
          .join(pub.group_by('cod_catmat').agg(pl.col('ano').min().alias('primeiro_ano'), pl.col('ano').max().alias('ultimo_ano'),
                                               pl.col('desc_item').drop_nulls().last().alias('descricao_bps')), on='cod_catmat')
          .sort(['compras', 'cod_catmat'], descending=[True, False]))
print(f'Códigos fora das classes de saúde: {fora.height:,} | compras: {fora["compras"].sum():,} '
      f'= {fora["compras"].sum() / pub.height:.2%} da base | classes: {fora["classe"].n_unique()}')
print('Origem da classe desses códigos:'); print(fora.group_by('origem_classe').len())
print(fora.group_by('classe', 'nome_classe').agg(pl.len().alias('codigos'), pl.col('compras').sum()).sort('compras', descending=True).head(15))
fora.write_csv(PASTA_AUDITORIA / '03g_codigos_fora_classes_saude.csv')
print('\nRegistro: Auditoria/03g_codigos_fora_classes_saude.csv (lista para conferência; nenhuma base foi alterada)')

Códigos fora das classes de saúde: 144 | compras: 2,167 = 0.26% da base | classes: 47
Origem da classe desses códigos:
shape: (1, 2)
┌─────────────────────────────┬─────┐
│ origem_classe               ┆ len │
│ ---                         ┆ --- │
│ str                         ┆ u32 │
╞═════════════════════════════╪═════╡
│ catalogo, consulta out/2026 ┆ 144 │
└─────────────────────────────┴─────┘
shape: (15, 4)
┌────────┬─────────────────────────────────┬─────────┬─────────┐
│ classe ┆ nome_classe                     ┆ codigos ┆ compras │
│ ---    ┆ ---                             ┆ ---     ┆ ---     │
│ str    ┆ str                             ┆ u32     ┆ u32     │
╞════════╪═════════════════════════════════╪═════════╪═════════╡
│ 7510   ┆ ARTIGOS PARA ESCRITÓRIO         ┆ 9       ┆ 390     │
│ 8105   ┆ SACOS E BOLSAS                  ┆ 19      ┆ 253     │
│ 4240   ┆ EQUIPAMENTO PARA SEGURANÇA E S… ┆ 11      ┆ 204     │
│ 8530   ┆ ARTIGOS PARA HIGIENE PESSOAL    ┆ 9       ┆ 167     │
│